# Aprendendo tcpdump na prática

Este notebook roda dentro de um container Linux, então o `tcpdump` pode ser usado de forma parecida com um servidor Linux real.

Vamos: (1) ver as interfaces disponíveis, (2) capturar tráfego gerado por nós mesmos, (3) interpretar a saída e (4) usar `scapy` para visualizar os pacotes de forma mais estruturada.


## Ver interfaces de rede disponíveis dentro do container

In [22]:
!ip addr


1: lo: <LOOPBACK,UP,LOWER_UP> mtu 65536 qdisc noqueue state UNKNOWN group default qlen 1000
    link/loopback 00:00:00:00:00:00 brd 00:00:00:00:00:00
    inet 127.0.0.1/8 scope host lo
       valid_lft forever preferred_lft forever
    inet6 ::1/128 scope host 
       valid_lft forever preferred_lft forever
2: eth0@if26: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 1500 qdisc noqueue state UP group default 
    link/ether 9a:3a:92:66:1e:a2 brd ff:ff:ff:ff:ff:ff link-netnsid 0
    inet 172.18.0.2/16 brd 172.18.255.255 scope global eth0
       valid_lft forever preferred_lft forever


Repare que dentro do container normalmente existe `lo` (loopback) e `eth0` (interface virtual criada pelo Docker). Neste notebook vamos usar essas interfaces para capturar pacotes.


2. Captura básica: 10 pacotes

In [23]:
!timeout 10 tcpdump -i eth0 -c 10 -nn || true


tcpdump: verbose output suppressed, use -v[v]... for full protocol decode
listening on eth0, link-type EN10MB (Ethernet), snapshot length 262144 bytes
18:01:35.264656 IP 172.18.0.2.8888 > 172.18.0.1.47326: Flags [P.], seq 2810084357:2810085023, ack 3432848988, win 552, options [nop,nop,TS val 2753680891 ecr 388781378], length 666
18:01:35.264714 IP 172.18.0.1.47326 > 172.18.0.2.8888: Flags [.], ack 666, win 6893, options [nop,nop,TS val 388781520 ecr 2753680891], length 0
18:01:35.366895 IP 172.18.0.2.8888 > 172.18.0.1.47326: Flags [P.], seq 666:1507, ack 1, win 552, options [nop,nop,TS val 2753680993 ecr 388781520], length 841
18:01:35.366938 IP 172.18.0.1.47326 > 172.18.0.2.8888: Flags [.], ack 1507, win 6916, options [nop,nop,TS val 388781622 ecr 2753680993], length 0
18:01:35.469486 IP 172.18.0.2.8888 > 172.18.0.1.47326: Flags [P.], seq 1507:2327, ack 1, win 552, options [nop,nop,TS val 2753681096 ecr 388781622], length 820
18:01:35.469564 IP 172.18.0.1.47326 > 172.18.0.2.8888: Fla

Nesse momento pode aparecer pouca coisa, porque talvez ninguém esteja gerando tráfego. Para facilitar, vamos capturar enquanto geramos tráfego de propósito.


## Gerando Tráfego

In [14]:
%%bash
rm -f /tmp/captura.pcap

timeout --signal=INT 8 tcpdump -i any -nn -w /tmp/captura.pcap &
TCPDUMP_PID=$!

sleep 1
curl -s https://example.com > /dev/null
sleep 2

wait $TCPDUMP_PID || true

echo "--- Leitura da captura ---"
tcpdump -r /tmp/captura.pcap -nn || true


tcpdump: data link type LINUX_SLL2
tcpdump: listening on any, link-type LINUX_SLL2 (Linux cooked v2), snapshot length 262144 bytes
77 packets captured
122 packets received by filter
0 packets dropped by kernel


--- Leitura da captura ---


reading from file /tmp/captura.pcap, link-type LINUX_SLL2 (Linux cooked v2), snapshot length 262144


17:36:44.051444 lo    In  IP 127.0.0.1.46297 > 127.0.0.1.50208: Flags [P.], seq 1573603159:1573603739, ack 1991270103, win 512, options [nop,nop,TS val 1976238700 ecr 1976238666], length 580
17:36:44.051479 lo    In  IP 127.0.0.1.50208 > 127.0.0.1.46297: Flags [.], ack 580, win 600, options [nop,nop,TS val 1976238700 ecr 1976238700], length 0
17:36:44.051498 lo    In  IP 127.0.0.1.46297 > 127.0.0.1.42028: Flags [P.], seq 385519994:385520574, ack 3641708015, win 512, options [nop,nop,TS val 1976238700 ecr 1976238666], length 580
17:36:44.051503 lo    In  IP 127.0.0.1.42028 > 127.0.0.1.46297: Flags [.], ack 580, win 600, options [nop,nop,TS val 1976238700 ecr 1976238700], length 0
17:36:44.052025 lo    In  IP 127.0.0.1.46297 > 127.0.0.1.50208: Flags [P.], seq 580:1221, ack 1, win 512, options [nop,nop,TS val 1976238700 ecr 1976238700], length 641
17:36:44.052064 lo    In  IP 127.0.0.1.50208 > 127.0.0.1.46297: Flags [.], ack 1221, win 595, options [nop,nop,TS val 1976238700 ecr 1976238700

## Interpretando a saída

Cada linha tem o formato:

`horário IP origem.porta > IP destino.porta: Flags [...], seq X, ack Y, win Z, length N`

No tráfego TCP é comum observar:

1. `Flags [S]` → SYN (cliente inicia conexão)
2. `Flags [S.]` → SYN-ACK (servidor responde)
3. `Flags [.]` → ACK (cliente confirma)

Depois disso, em conexões HTTPS, ocorre o handshake TLS e então a troca de dados.


In [15]:
%%bash
rm -f /tmp/https.pcap

timeout --signal=INT 6 tcpdump -i any -nn 'tcp port 443' -w /tmp/https.pcap &
TCPDUMP_PID=$!

sleep 1
curl -s https://example.com > /dev/null
sleep 2

wait $TCPDUMP_PID || true

echo "--- Leitura da captura HTTPS ---"
tcpdump -r /tmp/https.pcap -nn || true


tcpdump: data link type LINUX_SLL2
tcpdump: listening on any, link-type LINUX_SLL2 (Linux cooked v2), snapshot length 262144 bytes
32 packets captured
32 packets received by filter
0 packets dropped by kernel


--- Leitura da captura HTTPS ---


reading from file /tmp/https.pcap, link-type LINUX_SLL2 (Linux cooked v2), snapshot length 262144


17:37:01.020461 eth0  Out IP 172.18.0.2.57092 > 104.20.23.154.443: Flags [S], seq 2776519128, win 64240, options [mss 1460,sackOK,TS val 1648992000 ecr 0,nop,wscale 7], length 0
17:37:01.040906 eth0  In  IP 104.20.23.154.443 > 172.18.0.2.57092: Flags [S.], seq 2956222344, ack 2776519129, win 29184, options [mss 1460,sackOK,TS val 8738426 ecr 1648992000,nop,wscale 7], length 0
17:37:01.040950 eth0  Out IP 172.18.0.2.57092 > 104.20.23.154.443: Flags [.], ack 1, win 502, options [nop,nop,TS val 1648992020 ecr 8738426], length 0
17:37:01.088677 eth0  Out IP 172.18.0.2.57092 > 104.20.23.154.443: Flags [P.], seq 1:518, ack 1, win 502, options [nop,nop,TS val 1648992068 ecr 8738426], length 517
17:37:01.089899 eth0  In  IP 104.20.23.154.443 > 172.18.0.2.57092: Flags [.], ack 518, win 223, options [nop,nop,TS val 8738475 ecr 1648992068], length 0
17:37:01.113306 eth0  In  IP 104.20.23.154.443 > 172.18.0.2.57092: Flags [.], seq 1:1421, ack 518, win 4096, options [nop,nop,TS val 8738498 ecr 1648

Aqui usamos um **filtro BPF** (Filtro Passa-Banda)(`tcp port 443`). O `tcpdump` aceita filtros como `tcp`, `udp`, `icmp`, `port 80`, `port 443`, `host`, `src`, `dst` e combinações com `and`/`or`.

Isso é importante porque capturar "tudo" em uma rede real pode gerar dados demais.


In [20]:
%%bash
rm -f /tmp/http.pcap

timeout --signal=INT 8 tcpdump -i any -nn -X 'tcp port 80' -w /tmp/http.pcap &
TCPDUMP_PID=$!

sleep 1
curl -s http://example.com > /dev/null
sleep 2

wait $TCPDUMP_PID || true

echo "--- Leitura da captura HTTP ---"
tcpdump -r /tmp/http.pcap -nn -X | head -120 || true


tcpdump: data link type LINUX_SLL2
tcpdump: listening on any, link-type LINUX_SLL2 (Linux cooked v2), snapshot length 262144 bytes
11 packets captured
13 packets received by filter
0 packets dropped by kernel


--- Leitura da captura HTTP ---


reading from file /tmp/http.pcap, link-type LINUX_SLL2 (Linux cooked v2), snapshot length 262144


17:39:33.320935 eth0  Out IP 172.18.0.2.34712 > 172.66.147.243.80: Flags [S], seq 1398413619, win 64240, options [mss 1460,sackOK,TS val 1168744016 ecr 0,nop,wscale 7], length 0
	0x0000:  4500 003c 25cb 4000 4006 28a7 ac12 0002  E..<%.@.@.(.....
	0x0010:  ac42 93f3 8798 0050 535a 1933 0000 0000  .B.....PSZ.3....
	0x0020:  a002 faf0 ec78 0000 0204 05b4 0402 080a  .....x..........
	0x0030:  45a9 9e50 0000 0000 0103 0307            E..P........
17:39:33.340346 eth0  In  IP 172.66.147.243.80 > 172.18.0.2.34712: Flags [S.], seq 1622933858, ack 1398413620, win 29184, options [mss 1460,sackOK,TS val 2095901064 ecr 1168744016,nop,wscale 7], length 0
	0x0000:  4500 003c f84e 0000 3f06 9723 ac42 93f3  E..<.N..?..#.B..
	0x0010:  ac12 0002 0050 8798 60bc 0162 535a 1934  .....P..`..bSZ.4
	0x0020:  a012 7200 48a1 0000 0204 05b4 0402 080a  ..r.H...........
	0x0030:  7cec e988 45a9 9e50 0103 0307            |...E..P....
17:39:33.340405 eth0  Out IP 172.18.0.2.34712 > 172.66.147.243.80: Flags [.], ack 

Com HTTP puro (sem TLS), é possível observar mais detalhes do conteúdo no hexdump. 

Isso serve para explicar por que HTTPS é importante.


## Visão estruturada com scapy

In [17]:
from pathlib import Path
from scapy.all import rdpcap

pcap = Path("/tmp/captura.pcap")

if not pcap.exists() or pcap.stat().st_size == 0:
    print("Arquivo de captura não encontrado. Rode a célula de captura primeiro.")
else:
    pacotes = rdpcap(str(pcap))
    for pkt in pacotes[:15]:
        print(pkt.summary())


CookedLinuxV2 / IP / TCP 127.0.0.1:46297 > 127.0.0.1:50208 PA / Raw
CookedLinuxV2 / IP / TCP 127.0.0.1:50208 > 127.0.0.1:46297 A
CookedLinuxV2 / IP / TCP 127.0.0.1:46297 > 127.0.0.1:42028 PA / Raw
CookedLinuxV2 / IP / TCP 127.0.0.1:42028 > 127.0.0.1:46297 A
CookedLinuxV2 / IP / TCP 127.0.0.1:46297 > 127.0.0.1:50208 PA / Raw
CookedLinuxV2 / IP / TCP 127.0.0.1:50208 > 127.0.0.1:46297 A
CookedLinuxV2 / IP / TCP 127.0.0.1:46297 > 127.0.0.1:42028 PA / Raw
CookedLinuxV2 / IP / TCP 127.0.0.1:42028 > 127.0.0.1:46297 A
CookedLinuxV2 / IP / TCP 172.18.0.2:8888 > 172.18.0.1:47268 PA / Raw
CookedLinuxV2 / IP / TCP 172.18.0.1:47268 > 172.18.0.2:8888 A
CookedLinuxV2 / IP / TCP 172.18.0.2:8888 > 172.18.0.1:47274 PA / Raw
CookedLinuxV2 / IP / TCP 172.18.0.1:47274 > 172.18.0.2:8888 A
CookedLinuxV2 / IP / TCP 172.18.0.2:8888 > 172.18.0.1:47274 PA / Raw
CookedLinuxV2 / IP / TCP 172.18.0.1:47274 > 172.18.0.2:8888 A
CookedLinuxV2 / IP / TCP 172.18.0.2:8888 > 172.18.0.1:47326 PA / Raw


Inspecionando 1 pacote específico camada por camada

In [24]:
if "pacotes" in globals() and len(pacotes) > 0:
    pacotes[0].show()
else:
    print("Nenhum pacote carregado. Rode a célula anterior primeiro.")


###[ cooked linux v2 ]###
  proto     = IPv4
  reserved  = 0
  ifindex   = 1
  lladdrtype= 0x304
  pkttype   = unicast
  lladdrlen = 6
  src       = b''
###[ IP ]###
     version   = 4
     ihl       = 5
     tos       = 0x0
     len       = 632
     id        = 21061
     flags     = DF
     frag      = 0
     ttl       = 64
     proto     = 6
     chksum    = 0xe838
     src       = 127.0.0.1
     dst       = 127.0.0.1
     \options   \
###[ TCP ]###
        sport     = 46297
        dport     = 50208
        seq       = 1573603159
        ack       = 1991270103
        dataofs   = 8
        reserved  = 0
        flags     = PA
        window    = 512
        chksum    = 0x6d
        urgptr    = 0
        options   = [('NOP', None), ('NOP', None), ('Timestamp', (1976238700, 1976238666))]
###[ Raw ]###
           load      = b'\x01\rstream.stderr\x01\t<IDS|MSG>\x01@141bdba594fcec4c2b70246135a8e87e3fa407bdb860c3ff5d93f1952e961b35\x01\xcf{"msg_id": "792f8a4a-c1907f91ac76f324d2da70c9_226

### Explicação da inspeção do pacote

A saída acima mostra o pacote separado por camadas. Diferente do `tcpdump`, que apresenta uma visão mais resumida em texto, o `scapy` permite visualizar os campos internos do pacote de forma organizada.

Normalmente aparecem camadas como:

- **Ether**: informações da camada de enlace, como endereços MAC.
- **IP**: informações da camada de rede, como IP de origem e IP de destino.
- **TCP/UDP/ICMP**: informações da camada de transporte ou controle, como portas e flags.

No caso de um pacote TCP, campos como `sport` e `dport` indicam as portas de origem e destino. Já o campo `flags` mostra o estado da comunicação.

<table style="margin-left: 0; margin-right: auto;">
  <thead>
    <tr>
      <th>Flag</th>
      <th>Nome</th>
      <th>Significado</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>S</code></td>
      <td>SYN</td>
      <td>tentativa de iniciar conexão</td>
    </tr>
    <tr>
      <td><code>SA</code></td>
      <td>SYN + ACK</td>
      <td>resposta aceitando iniciar conexão</td>
    </tr>
    <tr>
      <td><code>A</code></td>
      <td>ACK</td>
      <td>confirmação de recebimento</td>
    </tr>
    <tr>
      <td><code>PA</code></td>
      <td>PSH + ACK</td>
      <td>envio de dados em uma conexão já estabelecida</td>
    </tr>
    <tr>
      <td><code>F</code></td>
      <td>FIN</td>
      <td>encerramento normal da conexão</td>
    </tr>
    <tr>
      <td><code>R</code></td>
      <td>RST</td>
      <td>reset ou interrupção da conexão</td>
    </tr>
  </tbody>
</table>

Um arquivo PCAP não é apenas uma lista de linhas de texto. Ele contém pacotes reais, compostos por várias camadas e campos técnicos.as e campos técnicos.cos. 

## Exercícios sugeridos

- Troque o filtro para `icmp` e rode `ping` para ver o formato dos pacotes ICMP.
- Capture as portas 80 e 443 ao mesmo tempo com `'tcp port 80 or tcp port 443'`.
- Use `-v`, `-vv` e `-vvv` no `tcpdump` e compare o nível de detalhe.
